### Bibliotecas necessárias.

In [40]:
# Instalar ou atualizar biblioteca necessária.
!pip install -q -U datasets huggingface_hub openai pathlib

# Importar bibliotecas necessárias
import json
import re
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from google import genai
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset
from openai import OpenAI

### Funções que povoam os Dataframes com as questões e respostas disponíveis no github.

In [41]:
# Resgata o segredo cadastrado com o nome para o hugging face
def logon_hugging_face(hugging_api_key):
  try:
    # Recuperar valor da chave criada no huggin face e cadastrada no Secrets do Google Colab.
    hf_token = userdata.get(hugging_api_key)
    # Realiza o login no Hugging Face
    login(token=hf_token)
  except Exception as e:
    print(f"Erro ao recuperar token: {e}")

# Povoar dataframe de questões do huggin face.
def load_questions(dataset_id):
  # Carrega o dataset (geralmente ele vem dividido em 'train', 'test', etc.)
  ds = load_dataset(dataset_id)

  # Converte uma partição específica 'train' para um dataframe.
  df = pd.DataFrame(ds['train'])

  # Inserir uma coluna para enumerar as questões, com contagem a partir do número 1, uma vez que a contagem de linhas do python é a partir do 0.
  df.insert(0, 'num', range(1, len(df)+1))

  # retorno da Dataframe.
  return df

# Dataframe com um subconjunto das perguntas e linhas guias.
# O índice do Dataframe começa do 0 (zero), portanto, a linha 1 é a 0, a 2 é a 1, etc.
# iloc seleciona um intervalo fechado à esquerda e aberto à direita
def load_my_close_questions(df_close_questions, question_min, question_max):
  # Remover 1 und dando o descontando, uma vez que o valor passado é o número da questão para o ser humano
  # e não a contagem do python.
  question_min -= 1
  df = df_close_questions.iloc[question_min:question_max].copy()
  return df

# Função para submeter uma questão à IA.
def question_submit(client_ai, model_id, question, question_type, alternatives):
    prompt_usuario = f"""
    Pessoa:
    Atue como um specialista Jurídico. Analise a questão da OAB abaixo:

    tipo_de_questao: {question_type}
    Questão: {question}
    Alternativas: {alternatives}

    Tarefas:
    1. Responda a questão escolhendo a letra correta (A, B, C ou D).
    2. Classifique a área de direito, com base na informação tipo_de_questao que pode está em inglês,
       e se for preciso traduza-a, para classificar a área.

    Responda no formato JSON somente com os dados que seguem:
    {{
      "resposta": "LETRA",
      "area": "NOME_DA_AREA"
    }}
    """
    # Início da medição de tempo
    start_time = time.time()

    # Check if the client is an OpenAI-compatible client (like from openai library, Groq, OpenRouter)
    if isinstance(client_ai, OpenAI):
      chat = client_ai.chat.completions.create(
          messages=[{"role": "user", "content": prompt_usuario}],
          model=model_id,
          response_format={"type": "json_object"}
      )
      return json.loads(chat.choices[0].message.content)
    # Check if the client is a Google Generative AI client
    elif isinstance(client_ai, genai.Client):
      response = client_ai.models.generate_content(
        model=model_id,
        contents=prompt_usuario,
        config=
          {
            "temperature": 0.1,  # Conservador.
          }
      )
      # The prompt asks for JSON, so we expect response.text to be a JSON string
      # We need to extract the text content correctly from the response object
      response_text = response.candidates[0].content.parts[0].text

      # Fim da medição de tempo
      end_time = time.time()

      # Calcula a duração em milisegundos
      duration_ms = (end_time - start_time) * 1000

      # Use regex for robust JSON extraction
      # This regex attempts to find either a JSON object within a markdown block (```json...```)
      # or a standalone JSON object.
      # The `re.DOTALL` flag allows '.' to match newline characters.
      json_match = re.search(r"```json\s*(\{[\s\S]*?\})\s*```|(\{[\s\S]*\})", response_text, re.DOTALL)

      json_string = None
      if json_match:
          # If the first group (markdown block) matched, use it. Otherwise, use the second group (standalone JSON).
          if json_match.group(1):
              json_string = json_match.group(1)
          elif json_match.group(2):
              json_string = json_match.group(2)

      if json_string:
          result = json.loads(json_string)
          result["time"] = duration_ms
      else:
          raise ValueError(f"Não foi possível extrair um JSON válido da resposta do modelo: {response_text}")
    else:
      raise ValueError("Unsupported AI client type provided.")

# Comparar resposta com gabarito.
def compare_answers(response, answer_key):
  if response == answer_key:
    return 'Certo'
  else:
    return 'Errado'

### Dataset do hugging face

In [42]:
categorias = {
  'ETHICS': 'Ética',
  'INTERNATIONAL': 'Internacional',
  'CONSTITUTIONAL': 'Constitucional',
  'BUSINESS': 'Negócios',
  'CONSUMER': 'Consumidor',
  'CIVIL': 'Civil',
  'CIVIL-PROCEDURE': 'Processo Civil',
  'ADMINISTRATIVE': 'Administrativo',
  'TAXES': 'Impostos',
  'LABOUR': 'Trabalho',
  'LABOUR-PROCEDURE': 'Processo Trabalhista',
  'ENVIRONMENTAL': 'Meio Ambiente',
  'CRIMINAL': 'Criminal',
  'CRIMINAL-PROCEDURE': 'Processo Criminal',
  'CHILDREN': 'Crianças',
  'HUMAN-RIGHTS': 'Direitos Humanos',
  'PHILOSOPHY': 'Filosofia',
  'PHILOSOPY': 'Filosofia'
}

# Chave salva no Secrets do Google Colab.
hugging_api_key = 'hugging_colab'
logon_hugging_face(hugging_api_key)

# Meu sub-grupo de questões e respostas.
# Subconjunto das questões: 319 a 424.
question_min = 319
question_max = 424

# Povoar dataframe de questões do huggin face.
dataset_id = 'eduagarcia/oab_exams'
df_questions = load_questions(dataset_id)

# Traduzir categorias do inglës ao português,
df_questions['category'] = df_questions['question_type'].map(categorias)

# Colunas selecionar para projeção no Dataframe.
columns=['num', 'exam_id', 'exam_year', 'category', 'question', 'choices', 'answerKey']
df_close_questions = df_questions[columns]
df_close_questions.columns = ['num', 'edition', 'year', 'category', 'question', 'choices', 'answer']
# Criar um dataset com meu subconjunto de questões.
df_my_close_questions = load_my_close_questions(df_close_questions, question_min, question_max)

### Dataframe das questões fechadas.

In [43]:
df_close_questions

,num,edition,year,category,question,choices,answer
0,1,2010-01,2010,Ética,Júlio e Lauro constituíram o mesmo advogado pa...,"{'text': ['optar, com prudência e discerniment...",A
1,2,2010-01,2010,Ética,"Mário, advogado regularmente inscrito na OAB, ...",{'text': ['Ainda que se reabilite criminalment...,D
2,3,2010-01,2010,Ética,De acordo com o Estatuto da Advocacia e da OAB...,{'text': ['decisão não unânime proferida por c...,A
3,4,2010-01,2010,Ética,Assinale a opção correta de acordo com as disp...,{'text': ['O compromisso que o requerente à in...,A
4,5,2010-01,2010,Ética,"Acerca das infrações e sanções disciplinares, ...",{'text': ['Considere que uma advogada inscrita...,B
...,...,...,...,...,...,...,...
2205,2206,2018-25,2018,NaN,Silvio contratou você como advogado para ajuiz...,"{'text': ['O mandato, no caso, é válido e os p...",A
2206,2207,2018-25,2018,NaN,Jéssica trabalhou na sociedade empresária Móve...,{'text': ['A sociedade empresária está correta...,C
2207,2208,2018-25,2018,NaN,Em sede de reclamações trabalhista duas socied...,{'text': ['deixar de recolher o depósito recur...,B
2208,2209,2018-25,2018,NaN,Em reclamação trabalhista já na fase de execuç...,"{'text': ['Agiu corretamente o juiz, porque as...",B


### Salvar dataframe em formato jsonl.

In [44]:
def save_df_to_jsonl(df: pd.DataFrame, filename: str):
  try:
    # Converte o DataFrame para JSONL
    # orient='records' serializa cada linha como um objeto JSON
    # lines=True garante que cada objeto JSON esteja em uma nova linha (formato JSONL)
    # json_format='plain' evita o escape de caracteres não-ASCII, preservando a acentuação.
    #jsonl_output = df.to_json(orient='records', lines=True, json_format='plain')

    # Exportando o dataframe para JSONL de forma correta
    df.to_json(
      filename,
      orient='records',
      lines=True,
      force_ascii=False  # <--- Não converter para ascii.
    )
    print(f"DataFrame salvo com sucesso em {filename} no formato JSONL.")
  except Exception as e:
    print(f"Ocorreu um erro ao salvar o DataFrame em JSONL: {e}")

# Exportar os dados processados até aqui em .jsonl para ocasião de uso posterior.
def ExportDataFrameToJSONL(df: pd.DataFrame, file: str):
  from google.colab import drive

  # Mount Google Drive
  drive.mount('/content/drive')

  # Diretório base
  diretorio = '/content/drive/My Drive'

  # 2. Define the path for the .jsonl file in Google Drive
  output_path = diretorio / file

  try:
    # Converte o DataFrame para JSONL
    # orient='records' serializa cada linha como um objeto JSON
    # lines=True garante que cada objeto JSON esteja em uma nova linha (formato JSONL)
    df.to_json(
      file,
      orient='records',
      lines=True,
      force_ascii=False)
    print(f"DataFrame salvo com sucesso em {file} no formato JSONL.")
  except Exception as e:
    print(f"Ocorreu um erro ao salvar o DataFrame em JSONL: {e}")

# Descomntar a linha abaixo e executar sobre demanda.
#ExportDataFrameToJSONL(df_my_close_questions)

### Salvar arquivo.

In [51]:
# Salvar arquivo jsonl.
# Incluir a coluna com o Tipo de questão que pode ser Aberta ou Fechada.
df_questions = df_close_questions
df_questions['type'] = 'Fechada'
save_df_to_jsonl(df_questions, 'close_questions.jsonl')

,num,edition,year,category,question,choices,answer,type
0,1,2010-01,2010,Ética,Júlio e Lauro constituíram o mesmo advogado pa...,"{'text': ['optar, com prudência e discerniment...",A,Fechada
1,2,2010-01,2010,Ética,"Mário, advogado regularmente inscrito na OAB, ...",{'text': ['Ainda que se reabilite criminalment...,D,Fechada
2,3,2010-01,2010,Ética,De acordo com o Estatuto da Advocacia e da OAB...,{'text': ['decisão não unânime proferida por c...,A,Fechada
3,4,2010-01,2010,Ética,Assinale a opção correta de acordo com as disp...,{'text': ['O compromisso que o requerente à in...,A,Fechada
4,5,2010-01,2010,Ética,"Acerca das infrações e sanções disciplinares, ...",{'text': ['Considere que uma advogada inscrita...,B,Fechada
...,...,...,...,...,...,...,...,...
2205,2206,2018-25,2018,NaN,Silvio contratou você como advogado para ajuiz...,"{'text': ['O mandato, no caso, é válido e os p...",A,Fechada
2206,2207,2018-25,2018,NaN,Jéssica trabalhou na sociedade empresária Móve...,{'text': ['A sociedade empresária está correta...,C,Fechada
2207,2208,2018-25,2018,NaN,Em sede de reclamações trabalhista duas socied...,{'text': ['deixar de recolher o depósito recur...,B,Fechada
2208,2209,2018-25,2018,NaN,Em reclamação trabalhista já na fase de execuç...,"{'text': ['Agiu corretamente o juiz, porque as...",B,Fechada


### Listar as diferentes categorias

In [46]:
df_close_questions['category'].unique()

array(['Ética', 'Internacional', 'Constitucional', 'Negócios',
       'Consumidor', 'Civil', 'Processo Civil', 'Administrativo',
       'Impostos', 'Trabalho', 'Processo Trabalhista', 'Meio Ambiente',
       'Criminal', 'Processo Criminal', 'Crianças', 'Direitos Humanos',
       'Filosofia', nan], dtype=object)

In [53]:
df_close_questions

,num,edition,year,category,question,choices,answer,type,question_ref_count,choices_ref_count,total_ref_count,level
0,1,2010-01,2010,Ética,Júlio e Lauro constituíram o mesmo advogado pa...,"{'text': ['optar, com prudência e discerniment...",A,Fechada,0,0,0,1
1,2,2010-01,2010,Ética,"Mário, advogado regularmente inscrito na OAB, ...",{'text': ['Ainda que se reabilite criminalment...,D,Fechada,0,0,0,1
2,3,2010-01,2010,Ética,De acordo com o Estatuto da Advocacia e da OAB...,{'text': ['decisão não unânime proferida por c...,A,Fechada,0,1,1,2
3,4,2010-01,2010,Ética,Assinale a opção correta de acordo com as disp...,{'text': ['O compromisso que o requerente à in...,A,Fechada,0,1,1,2
4,5,2010-01,2010,Ética,"Acerca das infrações e sanções disciplinares, ...",{'text': ['Considere que uma advogada inscrita...,B,Fechada,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
2205,2206,2018-25,2018,NaN,Silvio contratou você como advogado para ajuiz...,"{'text': ['O mandato, no caso, é válido e os p...",A,Fechada,0,0,0,1
2206,2207,2018-25,2018,NaN,Jéssica trabalhou na sociedade empresária Móve...,{'text': ['A sociedade empresária está correta...,C,Fechada,0,0,0,1
2207,2208,2018-25,2018,NaN,Em sede de reclamações trabalhista duas socied...,{'text': ['deixar de recolher o depósito recur...,B,Fechada,0,0,0,1
2208,2209,2018-25,2018,NaN,Em reclamação trabalhista já na fase de execuç...,"{'text': ['Agiu corretamente o juiz, porque as...",B,Fechada,0,1,1,2
